In [19]:
import subprocess, sys, os, time, glob, json, shutil
 
print("=== VERIFICAÇÃO DE AMBIENTE ===")
 
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "GPU não detectada!\n"
        "Ative em: Settings (painel direito) → Accelerator → GPU T4 x2"
    )
 
import psutil
print(f"RAM     : {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"Python  : {sys.version.split()[0]}")
 

=== VERIFICAÇÃO DE AMBIENTE ===
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
RAM     : 33.7 GB
Python  : 3.12.13


In [20]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Instalar dependências
# ═══════════════════════════════════════════════════════════
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow", "mlflow", "imbalanced-learn",
    "shap", "tqdm", "scikit-learn", "xgboost", "boruta"
], check=True)
print("Dependências instaladas.")
 

Dependências instaladas.


In [21]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — Clonar repositório
# ═══════════════════════════════════════════════════════════
REPO_DIR = "/kaggle/working/ascon"
 
if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "https://github.com/K1nginthen0rth/ascon.git"],
        capture_output=True, text=True,
        cwd="/kaggle/working"
    )
    print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError("Falha ao clonar repositório.")
    print("Repositório clonado.")
else:
    print("Repositório já existe — atualizando...")
    result = subprocess.run(
        ["git", "pull"],
        capture_output=True, text=True,
        cwd=REPO_DIR
    )
    print(result.stdout.strip())

Repositório já existe — atualizando...
Already up to date.


In [22]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — Criar symlinks para os dados (idempotente)
# ═══════════════════════════════════════════════════════════
DATA_DIR = "/kaggle/input/datasets/nycolaswenderson/lwc-ml-dataset"

os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)

links = {
    f"{REPO_DIR}/data/processed/keyholdout_2class_60k_v1.parquet":
        f"{DATA_DIR}/keyholdout_2class_60k_v1.parquet",
    f"{REPO_DIR}/data/processed/keyholdout_2class_60k_v1_features.parquet":
        f"{DATA_DIR}/keyholdout_2class_60k_v1_features.parquet",
}

for link, target in links.items():
    if os.path.islink(link):
        os.remove(link)
    if not os.path.exists(link):
        os.symlink(target, link)
        print(f"Symlink criado: {os.path.basename(link)}")
    else:
        print(f"Já existe: {os.path.basename(link)}")

Symlink criado: keyholdout_2class_60k_v1.parquet
Symlink criado: keyholdout_2class_60k_v1_features.parquet


In [23]:
import subprocess
result = subprocess.run(
    ["git", "pull"],
    capture_output=True, text=True,
    cwd="/kaggle/working/ascon"
)
print(result.stdout)
print(result.stderr)

Already up to date.




In [24]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — Configurar paths e forçar Sessão 2 (Caminho D)
# ═══════════════════════════════════════════════════════════
CKPT_DIR    = f"{REPO_DIR}/checkpoints"
REPORTS_DIR = f"{REPO_DIR}/reports"
MLFLOW_DIR  = f"{REPO_DIR}/mlruns"
GCP_BUCKET  = "gs://lwc-ml-checkpoints-nick"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MLFLOW_DIR, exist_ok=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Forçar Sessão 2 — B e C já concluídos, rodar só D
SESSAO = "1"
print(">>> SESSÃO 2 — Caminho D <<<")

>>> SESSÃO 2 — Caminho D <<<


In [25]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — Limpar checkpoints antigos do smoke test
# ═══════════════════════════════════════════════════════════
smoke_ckpt = f"{REPO_DIR}/reports/smoke_test"
if os.path.exists(smoke_ckpt):
    shutil.rmtree(smoke_ckpt)
    print(f"Removido: {smoke_ckpt}")
else:
    print("Sem checkpoints antigos do smoke test.")
 

Removido: /kaggle/working/ascon/reports/smoke_test


In [26]:
import os

link = f"{REPO_DIR}/data/processed/keyholdout_2class_60k_v1.parquet"
print("Link existe:", os.path.exists(link))
print("É symlink:", os.path.islink(link))
print("Aponta para:", os.readlink(link) if os.path.islink(link) else "N/A")
print("Target existe:", os.path.exists(os.readlink(link)) if os.path.islink(link) else "N/A")

Link existe: True
É symlink: True
Aponta para: /kaggle/input/datasets/nycolaswenderson/lwc-ml-dataset/keyholdout_2class_60k_v1.parquet
Target existe: True


In [27]:
import os
print(os.listdir("/kaggle/input/datasets/nycolaswenderson/"))

['lwc-ml-dataset']


In [28]:
"""# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — Smoke test (~5 min)
# ═══════════════════════════════════════════════════════════
print("=== SMOKE TEST ===")
 
result = subprocess.run([
    sys.executable, f"{REPO_DIR}/scripts/smoke_test.py",
], capture_output=True, text=True, cwd=REPO_DIR)
 
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Smoke test falhou. Corrija antes de continuar.")
 
print("Smoke test OK — prosseguindo.\n")"""

'# ═══════════════════════════════════════════════════════════\n# CÉLULA 7 — Smoke test (~5 min)\n# ═══════════════════════════════════════════════════════════\nprint("=== SMOKE TEST ===")\n \nresult = subprocess.run([\n    sys.executable, f"{REPO_DIR}/scripts/smoke_test.py",\n], capture_output=True, text=True, cwd=REPO_DIR)\n \nprint(result.stdout[-3000:])\nif result.returncode != 0:\n    print("STDERR:", result.stderr[-2000:])\n    raise RuntimeError("Smoke test falhou. Corrija antes de continuar.")\n \nprint("Smoke test OK — prosseguindo.\n")'

In [29]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 8 — SESSÃO 1: Experimentos B e C
# ═══════════════════════════════════════════════════════════
if SESSAO == "1":
 
    # ── Caminho B — CNN1D ──
    print("=" * 60)
    print("  EXPERIMENTO B — CNN1D")
    print("=" * 60)
    t0 = time.time()
 
    result = subprocess.run([
        sys.executable, f"{REPO_DIR}/scripts/run_cnn_experiments_60k.py",
        "--mode",           "cnn1d",
        "--checkpoint-dir", f"{CKPT_DIR}/caminhoB",
        "--resume",
    ], cwd=REPO_DIR)
 
    elapsed_b = time.time() - t0
    print(f"\nCaminho B concluído em {elapsed_b/3600:.2f}h "
          f"(returncode={result.returncode})")
 
    # ── Caminho C — CNN2D ──
    print("\n" + "=" * 60)
    print("  EXPERIMENTO C — CNN2D (co-ocorrência 256×256)")
    print("=" * 60)
    t0 = time.time()
 
    result = subprocess.run([
        sys.executable, f"{REPO_DIR}/scripts/run_cnn_experiments_60k.py",
        "--mode",           "cnn2d",
        "--checkpoint-dir", f"{CKPT_DIR}/caminhoC",
        "--resume",
    ], cwd=REPO_DIR)
 
    elapsed_c = time.time() - t0
    print(f"\nCaminho C concluído em {elapsed_c/3600:.2f}h "
          f"(returncode={result.returncode})")

  EXPERIMENTO B — CNN1D

  Experimento CNN — Ascon vs GIFT-COFB (60k, 64KB CT)
  CNN1D max_len=65552  CNN2D co-ocorrência 256×256
  device=cuda  mode=cnn1d  resume=True
  batch=64  epochs≤30  patience=5
  checkpoint_dir=checkpoints/caminhoB
  MLflow=ativo

  Carregando keyholdout_2class_60k_v1.parquet  (memory_map=True)...
  TrainVal: 48,000 amostras | 240 chaves
  Test    : 12,000 amostras | 60 chaves
  Classes : {'Ascon-AEAD128': 0, 'GIFT-COFB': 1}

--- 5-fold GroupKFold CV ---
  Sem progresso salvo — iniciando do zero

  [Fold 1/5]  treino=192 chaves (38,400)  val=48 chaves (9,600)
    [CNN1D]  max_len=65552
       Iniciando do zero (fold 0 cnn1d)
       ep  1  train=0.7111  val=0.6934  val_f1=0.3365
       ep  2  train=0.6950  val=0.6936  val_f1=0.3333
       ep  3  train=0.6941  val=0.6938  val_f1=0.3333
       ep  4  train=0.6935  val=0.6951  val_f1=0.3333
       ep  5  train=0.6939  val=0.6933  val_f1=0.3344
       ep  6  train=0.6929  val=0.6936  val_f1=0.4770
       ep  7  tra

In [30]:
result = subprocess.run(
    ["git", "log", "--oneline", "-5"],
    capture_output=True, text=True,
    cwd="/kaggle/working/ascon"
)
print(result.stdout)

25dcf24 atualizacao padding
caeb842 feat(cnn): Caminho B usa CT completo (MAX_LEN_B=65552) + batch_size=8 no CNN1D
c49bf10 mh
d809a0d feat(hybrid): batch_size=8 no CNN1D quando CT longo (>=16384) no Caminho D
2f27ca1 fix(cnn): mover xb/yb para device em _train_fixed_epochs + device param



In [31]:
subprocess.run(["git", "pull"], cwd=REPO_DIR, capture_output=True)
print("Código atualizado.")

Código atualizado.


In [32]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 9 — BACKUP após B e C
# ═══════════════════════════════════════════════════════════
if SESSAO == "1":
    print("\n=== BACKUP GCP — após B e C ===")
    for folder in ["checkpoints", "reports", "mlruns"]:
        src = f"{REPO_DIR}/{folder}/"
        dst = f"{GCP_BUCKET}/{folder}/"
        if os.path.exists(f"{REPO_DIR}/{folder}"):
            r = subprocess.run([
                "gsutil", "-m", "rsync", "-r", src, dst
            ], capture_output=True, text=True)
            status = "OK" if r.returncode == 0 else "FALHOU"
            print(f"  {folder}: {status}")
        else:
            print(f"  {folder}: pasta não existe, pulando")
 
    print("\nBackup concluído.")
    print("\n" + "=" * 60)
    print("  SESSÃO 1 CONCLUÍDA")
    print("=" * 60)
    print("""
PRÓXIMOS PASSOS:
  1. Criar novo notebook no Kaggle
  2. Anexar dataset: lwc-ml-dataset
  3. Colar este mesmo notebook e rodar
     (vai restaurar B e C do GCP automaticamente)
  4. Vai detectar Sessão 2 e rodar só o Caminho D
""")
 
 


=== BACKUP GCP — após B e C ===
  checkpoints: FALHOU
  reports: FALHOU
  mlruns: FALHOU

Backup concluído.

  SESSÃO 1 CONCLUÍDA

PRÓXIMOS PASSOS:
  1. Criar novo notebook no Kaggle
  2. Anexar dataset: lwc-ml-dataset
  3. Colar este mesmo notebook e rodar
     (vai restaurar B e C do GCP automaticamente)
  4. Vai detectar Sessão 2 e rodar só o Caminho D



In [33]:
import glob
for f in glob.glob("/kaggle/working/**/*.pt", recursive=True) + \
         glob.glob("/kaggle/working/**/*.pkl", recursive=True):
    print(f)

/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_fold0.pt
/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_fold1.pt
/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_fold2.pt
/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_fold3.pt
/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_final.pt
/kaggle/working/ascon/checkpoints/caminhoC/cnn2d_fold4.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_fold2.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_fold0.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_fold4.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_final.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_fold3.pt
/kaggle/working/ascon/checkpoints/caminhoB/cnn1d_fold1.pt
/kaggle/working/ascon/checkpoints/caminhoC/_cv_progress.pkl
/kaggle/working/ascon/checkpoints/caminhoB/_cv_progress.pkl
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_cnn/_cv_cache_cnn2d.pkl
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_cnn/_final_cache_cnn2d.pkl
/kaggle/working/ascon/r

In [34]:
import shutil, os

os.makedirs("/kaggle/working/output_backup", exist_ok=True)
shutil.copytree(
    "/kaggle/working/ascon/checkpoints",
    "/kaggle/working/output_backup/checkpoints",
    dirs_exist_ok=True
)
print("Backup local feito.")

import glob
for f in glob.glob("/kaggle/working/output_backup/**/*", recursive=True):
    print(f)

Backup local feito.
/kaggle/working/output_backup/checkpoints
/kaggle/working/output_backup/checkpoints/caminhoC
/kaggle/working/output_backup/checkpoints/caminhoB
/kaggle/working/output_backup/checkpoints/caminhoC/_cv_progress.pkl
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_fold0.pt
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_fold1.pt
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_fold2.pt
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_fold3.pt
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_final.pt
/kaggle/working/output_backup/checkpoints/caminhoC/cnn2d_fold4.pt
/kaggle/working/output_backup/checkpoints/caminhoB/_cv_progress.pkl
/kaggle/working/output_backup/checkpoints/caminhoB/cnn1d_fold2.pt
/kaggle/working/output_backup/checkpoints/caminhoB/cnn1d_fold0.pt
/kaggle/working/output_backup/checkpoints/caminhoB/cnn1d_fold4.pt
/kaggle/working/output_backup/checkpoints/caminhoB/cnn1d_final.pt
/kaggle/working/output_backup/checkpoint

In [18]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 10 — SESSÃO 2: Experimento D (Híbrido)
# ═══════════════════════════════════════════════════════════
import subprocess, sys, time
if SESSAO == "2":
 
    print("=" * 60)
    print("  EXPERIMENTO D — HÍBRIDO")
    print("  CNN1D com CT completo (65.552 bytes) — GPU obrigatória")
    print("=" * 60)
    t0 = time.time()
 
    result = subprocess.run([
        sys.executable, f"{REPO_DIR}/scripts/run_hybrid_60k.py",
        "--checkpoint-dir", f"{CKPT_DIR}/caminhoD",
        "--resume",
    ], cwd=REPO_DIR)
 
    elapsed_d = time.time() - t0
    print(f"\nCaminho D concluído em {elapsed_d/3600:.2f}h "
          f"(returncode={result.returncode})")
 

  EXPERIMENTO D — HÍBRIDO
  CNN1D com CT completo (65.552 bytes) — GPU obrigatória

  Caminho D — Híbrido (60k, 64KB CT)
  [307D clássicas | latent CNN1D | latent CNN2D] → MI→mRMR→Boruta → RF/XGB
  Protocolo: 80/20 key-holdout + 5-fold GroupKFold CV
  CNN1D max_len (Caminho B ref.) : 4096 bytes
  CNN1D max_len (Caminho D)      : 65552 bytes
  resume=True  checkpoint_dir=checkpoints/caminhoD

  GPU disponivel — CNN1D usa CT completo (65552 bytes)

  Carregando features: keyholdout_2class_60k_v1_features.parquet
  Carregando ciphertexts: keyholdout_2class_60k_v1.parquet
  TrainVal : 48,000 | 240 chaves
  Test     : 12,000 | 60 chaves
  Classes  : {'Ascon-AEAD128': 0, 'GIFT-COFB': 1}
  Features clássicas: 307
  Vetor híbrido esperado: 947D (307 + 512 CNN1D + 128 CNN2D)

--- 5-fold GroupKFold CV ---
  Sem progresso salvo — iniciando do zero

  [Fold 1/5]  treino=192 chaves (38,400)  val=48 chaves (9,600)
    [CNN1D] max_len=65552
  Iniciando do zero (fold 0 1d)
    ep  1  train=0.7118  val

    ep  1  train=0.7002  val=2.6129  f1=0.3333
    ep  2  train=0.6944  val=0.7011  f1=0.3333
    ep  3  train=0.6940  val=0.9246  f1=0.3333
    ep  4  train=0.6938  val=0.6938  f1=0.3333
    ep  5  train=0.6938  val=0.6932  f1=0.3333
    ep  6  train=0.6935  val=4.0034  f1=0.3333
    ep  7  train=0.6937  val=0.7709  f1=0.3333
    ep  8  train=0.6936  val=0.8347  f1=0.3333
    ep  9  train=0.6933  val=0.7372  f1=0.3333
    ep 10  train=0.6935  val=0.6944  f1=0.3333
    early stop @ ep 10 (best 5)
    CNN2D ok: best_ep=5  t=1136.2s


    Vetor híbrido: 947D (307 clássicas + 512 CNN1D + 128 CNN2D)
    FS: 947 → 91 features  (357.9s)  (MI→91  mRMR→91  Boruta→91)
    RF          F1=0.4980  BalAcc=0.4980  t=165.0s
    XGBoost     F1=0.4960  BalAcc=0.4960  t=10.5s

  [Fold 4/5]  treino=192 chaves (38,400)  val=48 chaves (9,600)
    [CNN1D] max_len=65552
  Iniciando do zero (fold 3 1d)


In [20]:
import subprocess

result = subprocess.run([
    "bash", "-c",
    "cd /kaggle/working/ascon && "
    "zip -r /kaggle/working/resultados_caminhoD_final.zip "
    "reports/ checkpoints/caminhoD/_cv_cache.pkl checkpoints/caminhoD/_final_cache.pkl "
    "-x '*.pt'"
], capture_output=True, text=True)

print(result.stdout[-2000:])
print(result.stderr[-1000:])

	zip warning: name not matched: checkpoints/caminhoD/_cv_cache.pkl
	zip warning: name not matched: checkpoints/caminhoD/_final_cache.pkl
  adding: reports/ (stored 0%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/ (stored 0%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/cv_results.json (deflated 87%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/confusion_matrices/ (stored 0%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/_cv_cache.pkl (deflated 54%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/final_results.json (deflated 98%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/comparison_table.md (deflated 37%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/ckpts/ (stored 0%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/_final_cache.pkl (deflated 96%)
  adding: reports/keyholdout_2class_60k_v1_hybrid/mcnemar_table.md (deflated 34%)




In [21]:
import os
print(os.path.exists("/kaggle/working/resultados_caminhoD_final.zip"))
print(f"{os.path.getsize('/kaggle/working/resultados_caminhoD_final.zip') / 1e6:.1f} MB")

True
0.0 MB


In [22]:
import subprocess
result = subprocess.run(
    ["ls", "-la", "/kaggle/working/resultados_caminhoD_final.zip"],
    capture_output=True, text=True
)
print(result.stdout)

-rw-r--r-- 1 root root 19095 Jun 21 23:08 /kaggle/working/resultados_caminhoD_final.zip



In [23]:
import subprocess

result = subprocess.run([
    "bash", "-c",
    "cd /kaggle/working/ascon && "
    "zip -r /kaggle/working/resultados_caminhoB_C_final.zip "
    "reports/keyholdout_2class_60k_v1_cnn/ "
    "-x '*.pt'"
], capture_output=True, text=True)

print(result.stdout[-2000:])
print(result.stderr[-1000:])

	zip warning: name not matched: reports/keyholdout_2class_60k_v1_cnn/

zip error: Nothing to do! (/kaggle/working/resultados_caminhoB_C_final.zip)




In [27]:
import subprocess
result = subprocess.run(
    ["find", "/kaggle/working/ascon/reports/", "-maxdepth", "2"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

/kaggle/working/ascon/reports/
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/cv_results.json
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/confusion_matrices
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/_cv_cache.pkl
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/final_results.json
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/comparison_table.md
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/ckpts
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/_final_cache.pkl
/kaggle/working/ascon/reports/keyholdout_2class_60k_v1_hybrid/mcnemar_table.md




In [19]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 11 — BACKUP após D
# ═══════════════════════════════════════════════════════════
if SESSAO == "2":
    print("\n=== BACKUP GCP — após D ===")
    for folder in ["checkpoints", "reports", "mlruns"]:
        src = f"{REPO_DIR}/{folder}/"
        dst = f"{GCP_BUCKET}/{folder}/"
        if os.path.exists(f"{REPO_DIR}/{folder}"):
            r = subprocess.run([
                "gsutil", "-m", "rsync", "-r", src, dst
            ], capture_output=True, text=True)
            status = "OK" if r.returncode == 0 else "FALHOU"
            print(f"  {folder}: {status}")
        else:
            print(f"  {folder}: pasta não existe, pulando")
 
    print("\nBackup concluído.")
    print("\n" + "=" * 60)
    print("  SESSÃO 2 CONCLUÍDA — TODOS OS EXPERIMENTOS PRONTOS")
    print("=" * 60)


=== BACKUP GCP — após D ===


NameError: name 'GCP_BUCKET' is not defined

In [37]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 12 — Resumo de resultados
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  RESUMO DOS RESULTADOS")
print("=" * 60)
 
for caminho in ["caminhoB", "caminhoC", "caminhoD"]:
    pattern = f"{REPORTS_DIR}/{caminho}/**/*.json"
    files = glob.glob(pattern, recursive=True)
    if not files:
        print(f"\n  {caminho}: sem resultados ainda")
        continue
    print(f"\n  --- {caminho} ---")
    for f in sorted(files):
        try:
            with open(f) as fh:
                data = json.load(fh)
            print(f"    {os.path.basename(f)}:")
            for k, v in data.items():
                if isinstance(v, (int, float)) and \
                   any(m in k.lower() for m in ["f1", "acc", "ece"]):
                    print(f"      {k}: {v:.4f}")
        except Exception as e:
            print(f"    {os.path.basename(f)}: erro ({e})")


  RESUMO DOS RESULTADOS

  caminhoB: sem resultados ainda

  caminhoC: sem resultados ainda

  caminhoD: sem resultados ainda


In [8]:
import subprocess
result = subprocess.run(
    ["ls", "-la", "/kaggle/working/ascon"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

total 116
drwxr-xr-x 8 root root  4096 Jun 21 15:12 .
drwxr-xr-x 4 root root  4096 Jun 21 15:12 ..
-rw-r--r-- 1 root root   284 Jun 21 15:12 build_cffi.bat
-rw-r--r-- 1 root root   452 Jun 21 15:12 build_gift_cofb.bat
-rw-r--r-- 1 root root  6138 Jun 21 15:12 CLAUDE.md
-rw-r--r-- 1 root root   495 Jun 21 15:12 conftest.py
-rw-r--r-- 1 root root 42137 Jun 21 15:12 CONTEXTO_ARTIGO.md
drwxr-xr-x 3 root root  4096 Jun 21 15:12 data
drwxr-xr-x 2 root root  4096 Jun 21 15:12 docs
drwxr-xr-x 8 root root  4096 Jun 21 15:12 .git
-rw-r--r-- 1 root root    14 Jun 21 15:12 .gitattributes
-rw-r--r-- 1 root root   627 Jun 21 15:12 .gitignore
-rw-r--r-- 1 root root  1071 Jun 21 15:12 LICENSE
-rw-r--r-- 1 root root  1615 Jun 21 15:12 mrmr.py
-rw-r--r-- 1 root root   165 Jun 21 15:12 run_tests.bat
drwxr-xr-x 2 root root  4096 Jun 21 15:12 scripts
drwxr-xr-x 6 root root  4096 Jun 21 15:12 src
drwxr-xr-x 2 root root  4096 Jun 21 15:12 tests




In [9]:
import os

DATA_DIR = "/kaggle/input/datasets/nycolaswenderson/lwc-ml-dataset"
REPO_DIR = "/kaggle/working/ascon"

os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)

links = {
    f"{REPO_DIR}/data/processed/keyholdout_2class_60k_v1.parquet":
        f"{DATA_DIR}/keyholdout_2class_60k_v1.parquet",
    f"{REPO_DIR}/data/processed/keyholdout_2class_60k_v1_features.parquet":
        f"{DATA_DIR}/keyholdout_2class_60k_v1_features.parquet",
}

for link, target in links.items():
    if os.path.islink(link):
        os.remove(link)
    if not os.path.exists(link):
        os.symlink(target, link)
        print(f"Symlink criado: {os.path.basename(link)}")
    else:
        print(f"Já existe: {os.path.basename(link)}")

Symlink criado: keyholdout_2class_60k_v1.parquet
Symlink criado: keyholdout_2class_60k_v1_features.parquet


In [10]:
import os, json

# Configurar autenticação GCP via Kaggle Secret
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
gcp_key = secrets.get_secret("GCP_SA_KEY")

with open("/kaggle/working/gcp-sa-key.json", "w") as f:
    f.write(gcp_key)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/working/gcp-sa-key.json"

import subprocess
result = subprocess.run(
    ["gcloud", "auth", "activate-service-account", "--key-file=/kaggle/working/gcp-sa-key.json"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


Activated service account credentials for: [kaggle-sa@lwc-ml-nick.iam.gserviceaccount.com]



In [11]:
import subprocess

result = subprocess.run(
    ["gsutil", "ls", "-r", "gs://lwc-ml-checkpoints-nick/"],
    capture_output=True, text=True
)
print(result.stdout[:3000])
print("STDERR:", result.stderr[:1000])

gs://lwc-ml-checkpoints-nick/test/:
gs://lwc-ml-checkpoints-nick/test/teste.txt

STDERR: 


In [12]:
import subprocess, os

os.makedirs("/kaggle/working/test_backup", exist_ok=True)
with open("/kaggle/working/test_backup/teste2.txt", "w") as f:
    f.write("teste backup")

result = subprocess.run([
    "gsutil", "-m", "rsync", "-r",
    "/kaggle/working/test_backup/",
    "gs://lwc-ml-checkpoints-nick/test_backup/"
], capture_output=True, text=True)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

STDOUT: 
STDERR: 
both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Building synchronization state...
Starting synchronization...
Copying file:///kaggle/working/test_backup/teste2.txt [Content-Type=text/plain]...
/ [0/1 files][    0.0 B/   12.0 B]   0% Done                                    
AccessDeniedException: 403 The billing account for the owning project is disabled in state closed
CommandException: 1 files/objects could not be copied/removed.

Return code: 1


In [14]:
import os

REPO_DIR = "/kaggle/working/ascon"
CKPT_DIR = f"{REPO_DIR}/checkpoints"
SESSAO = "2"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(f"{CKPT_DIR}/caminhoD", exist_ok=True)

print("SESSAO:", SESSAO)
print("REPO_DIR:", REPO_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("Checkpoint dir caminhoD existe?", os.path.exists(f"{CKPT_DIR}/caminhoD"))

SESSAO: 2
REPO_DIR: /kaggle/working/ascon
CKPT_DIR: /kaggle/working/ascon/checkpoints
Checkpoint dir caminhoD existe? True


In [15]:
print("SESSAO:", SESSAO)
print("REPO_DIR:", REPO_DIR)
print("CKPT_DIR:", CKPT_DIR)

import os
print("Checkpoint dir existe?", os.path.exists(f"{CKPT_DIR}/caminhoD"))

SESSAO: 2
REPO_DIR: /kaggle/working/ascon
CKPT_DIR: /kaggle/working/ascon/checkpoints
Checkpoint dir existe? True
